# Google Stock - LSTM Family Comparison

Notebook này chạy các biến thể LSTM trên đúng một dataset để so sánh kết quả thực nghiệm.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "dataset"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
tf.random.set_seed(42)
np.random.seed(42)


In [ ]:
df = pd.read_csv(DATA_DIR / "google_stock.csv")
close_col = [c for c in df.columns if c.lower() == "close" or "close" in c.lower()][0]
series = df[close_col].to_numpy().astype("float32")
series = (series - series.min()) / (series.max() - series.min())
window = 30
X, y = [], []
for i in range(len(series) - window):
    X.append(series[i:i+window])
    y.append(series[i+window])
X = np.array(X)[..., None]
y = np.array(y)
split = int(0.8 * len(X))
x_train, x_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]
input_shape = x_train.shape[1:]


In [ ]:
def build_lstm(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64),
        layers.Dense(1)
    ])

def build_stacked_lstm(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=True),
        layers.LSTM(64),
        layers.Dense(1)
    ])

def build_bilstm(input_shape):
    return keras.Sequential([
        layers.Input(shape=input_shape),
        layers.Bidirectional(layers.LSTM(64)),
        layers.Dense(1)
    ])

def build_attention_lstm(input_shape):
    inputs = keras.Input(shape=input_shape)
    x = layers.LSTM(64, return_sequences=True)(inputs)
    attention = layers.Attention()([x, x])
    x = layers.GlobalAveragePooling1D()(attention)
    outputs = layers.Dense(1)(x)
    return keras.Model(inputs, outputs)

builders = {
    "StandardLSTM": build_lstm,
    "StackedLSTM": build_stacked_lstm,
    "BiLSTM": build_bilstm,
    "LSTM+Attention": build_attention_lstm,
}

results = []
for name, builder in builders.items():
    model = builder(input_shape)
    model.compile(optimizer='adam', loss='mse', metrics=['mae'])
    model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=5, batch_size=64, verbose=0)
    loss, mae = model.evaluate(x_test, y_test, verbose=0)
    results.append({"model": name, "test_mse": loss, "test_mae": mae})

results_df = pd.DataFrame(results).sort_values("test_mse")
results_df

sns.barplot(data=results_df, x="test_mse", y="model", palette="rocket")
plt.title("LSTM-family Comparison")
plt.show()
